## Set Up

In [53]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString


In [54]:
url_1 = 'https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/data/processed/sf_business_demographics_nhood_naics.parquet'
combined = pd.read_parquet(url_1)

In [55]:
url_2 = 'https://raw.githubusercontent.com/miaflynn/CYPLAN255-Final-Project/main/data/processed/sf_businesses_nhood_naics.parquet'
neighborhood_table = pd.read_parquet(url_2)

In [56]:
neighborhood_table

,nhood,naics_group,year,closed,opened,biz_stock
0,Western Addition,Manufacturing & Industrial,2016,8.0,24.0,168
1,Western Addition,Utilities & Construction,2016,4.0,7.0,168
2,Western Addition,Retail,2016,5.0,23.0,168
3,Western Addition,Personal Services,2016,3.0,4.0,168
4,Western Addition,Food & Entertainment,2016,4.0,15.0,168
...,...,...,...,...,...,...
2646,Bayview Hunters Point,Personal Services,2025,15.0,9.0,320
2647,Bayview Hunters Point,Utilities & Construction,2025,24.0,34.0,320
2648,Bayview Hunters Point,Service,2025,34.0,19.0,320
2649,Bayview Hunters Point,Food & Entertainment,2025,28.0,49.0,320


## San Francisco Business Openings vs Closings (2016–2025)

## COVID Impact vs Recovery by Neighborhood

In [57]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

df = combined.copy()
base_color = "lightgray"
highlight_color = "steelblue"

def get_plot_df(naics_group):
    if naics_group == 'All':
        return df.groupby('neighborhood')[['closure_rate', 'recovery_rate', 'closures_2020_21', 'openings_2022_25']].mean()
    return df[df['naics_group'] == naics_group].set_index('neighborhood')

def make_fig(naics_group, highlight):
    plot_df = get_plot_df(naics_group)

    x_mean, y_mean = plot_df["closure_rate"].mean(), plot_df["recovery_rate"].mean()
    x_min, x_max = plot_df["closure_rate"].min(), plot_df["closure_rate"].max()
    y_min, y_max = plot_df["recovery_rate"].min(), plot_df["recovery_rate"].max()
    x_pad = (x_max - x_min) * 0.08
    y_pad = (y_max - y_min) * 0.08

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=plot_df["closure_rate"],
        y=plot_df["recovery_rate"],
        mode="markers",
        text=plot_df.index,
        customdata=plot_df[["closures_2020_21", "openings_2022_25"]].values,
        hovertemplate="<b>%{text}</b><br>Closure rate: %{x:.3f}<br>Recovery rate: %{y:.3f}<extra></extra>",
        marker=dict(
            color=[highlight_color if n == highlight else base_color for n in plot_df.index],
            size=[14 if n == highlight else 10 for n in plot_df.index]
        ),
    ))

    fig.add_shape(type="line", x0=x_mean, x1=x_mean, y0=y_min, y1=y_max, line=dict(dash="dash", color="black"))
    fig.add_shape(type="line", x0=x_min, x1=x_max, y0=y_mean, y1=y_mean, line=dict(dash="dash", color="black"))

    for x, y, text in [
        (x_max - x_pad, y_max - y_pad, "High closure<br>High recovery"),
        (x_min + x_pad, y_max - y_pad, "Low closure<br>High recovery"),
        (x_min + x_pad, y_min + y_pad, "Low closure<br>Low recovery"),
        (x_max - x_pad, y_min + y_pad, "High closure<br>Low recovery"),
    ]:
        fig.add_annotation(x=x, y=y, text=text, showarrow=False,
                           font=dict(size=10, color="gray"), bgcolor="rgba(255,255,255,0.6)")

    fig.update_layout(title=f"COVID Impact vs Recovery — {naics_group}", height=500,
                      xaxis_title="Closure Rate (2020–2021)", yaxis_title="Recovery Rate (2022–2025)")
    return fig

sector_dd = widgets.Dropdown(options=['All'] + sorted(df['naics_group'].unique().tolist()), value='All', description='Sector:')
nhood_dd  = widgets.Dropdown(options=['None'] + sorted(df['neighborhood'].unique().tolist()), value='None', description='Highlight:')
out = widgets.Output()

def on_change(_):
    with out:
        out.clear_output(wait=True)
        display(make_fig(sector_dd.value, nhood_dd.value))

sector_dd.observe(on_change, names='value')
nhood_dd.observe(on_change, names='value')

with out:
    display(make_fig('All', 'None'))

display(widgets.HBox([sector_dd, nhood_dd]), out)

Output()

## Race/Ethnicity and Median Income by Neighborhood

In [58]:
import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import Dropdown, Output, VBox, HBox, SelectMultiple
from IPython.display import display

out = Output()

def plot_neighborhoods(neighborhoods):
    if not neighborhoods:
        return

    race_cols   = ["pct_white", "pct_black", "pct_asian", "pct_latina_o", "pct_other"]
    race_labels = ["White", "Black", "Asian/PI", "Latino", "Other"]
    colors = ["#378ADD", "#E87040", "#4CAF50", "#9C27B0"]

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [3, 1]})

    n = len(neighborhoods)
    bar_width = 0.8 / n

    for i, neighborhood in enumerate(neighborhoods):
        row = combined[combined['neighborhood'] == neighborhood].iloc[0]
        values = [row[c] * 100 for c in race_cols]
        x = [j + i * bar_width for j in range(len(race_cols))]
        ax.bar(x, values, width=bar_width, color=colors[i], label=neighborhood)

    ax.set_xticks([j + bar_width * (n - 1) / 2 for j in range(len(race_cols))])
    ax.set_xticklabels(race_labels)
    ax.set_ylabel("% of population")
    ax.set_ylim(0, 100)
    ax.legend()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax2.axis("off")
    y_pos = 0.9
    for i, neighborhood in enumerate(neighborhoods):
        row = combined[combined['neighborhood'] == neighborhood].iloc[0]
        income = row["median_income"]
        income_str = f"${income:,.0f}" if pd.notna(income) else "No data"
        ax2.text(0.5, y_pos, neighborhood, ha="center", fontsize=9, color="gray", transform=ax2.transAxes)
        ax2.text(0.5, y_pos - 0.08, income_str, ha="center", fontsize=14,
                 fontweight="bold", color=colors[i], transform=ax2.transAxes)
        y_pos -= 0.25

    fig.suptitle("Race/Ethnicity by Neighborhood", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

selector = SelectMultiple(
    options=sorted(combined['neighborhood'].unique().tolist()),
    value=['Sunset/Parkside', 'Bayview Hunters Point'],
    rows=8,
    description='Neighborhoods:'
)

def on_change(change):
    if len(change['new']) > 4:
        selector.value = change['new'][:4]
        return
    with out:
        out.clear_output(wait=True)
        plot_neighborhoods(list(change['new']))

selector.observe(on_change, names='value')



with out:
    plot_neighborhoods(list(selector.value))

display(HBox([selector, out]))

## Business Resilience by Neighborhood Demographics

In [59]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

def make_fig(naics_group):
    if naics_group == 'All':
        plot_df = combined.groupby('neighborhood')[['median_income', 'pct_poc', 'resilience']].mean().reset_index()
    else:
        plot_df = combined[combined['naics_group'] == naics_group].copy().reset_index()

    hover_income = (
        "<b>%{text}</b><br>"
        "Resilience: %{y:.3f}<br>"
        "Median Income: $%{x:,.0f}<br>"
        "<extra></extra>"
    )
    hover_poc = (
        "<b>%{text}</b><br>"
        "Resilience: %{y:.3f}<br>"
        "% POC: %{x:.1%}<br>"
        "<extra></extra>"
    )

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Resilience vs Median Income", "Resilience vs % People of Color"),
        horizontal_spacing=0.12
    )

    fig.add_trace(go.Scatter(
        x=plot_df['median_income'],
        y=plot_df['resilience'],
        mode='markers+text',
        text=plot_df['neighborhood'],
        textposition='top center',
        textfont=dict(size=7, color='gray'),
        hovertemplate=hover_income,
        marker=dict(
            color=plot_df['median_income'],
            colorscale='RdBu',
            size=10,
            showscale=True,
            colorbar=dict(title='Median Income', x=0.44),
            line=dict(width=1, color='white')
        ),
        showlegend=False
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=plot_df['pct_poc'],
        y=plot_df['resilience'],
        mode='markers+text',
        text=plot_df['neighborhood'],
        textposition='top center',
        textfont=dict(size=7, color='gray'),
        hovertemplate=hover_poc,
        marker=dict(
            color=plot_df['pct_poc'],
            colorscale='RdBu_r',
            size=10,
            showscale=True,
            colorbar=dict(title='% POC', x=1.02),
            line=dict(width=1, color='white')
        ),
        showlegend=False
    ), row=1, col=2)

    for col in [1, 2]:
        fig.add_shape(
            type='line', x0=0, x1=1, y0=0, y1=0,
            xref=f"x{'' if col==1 else col} domain",
            yref=f"y{'' if col==1 else col}",
            line=dict(dash='dash', color='black', width=1),
            row=1, col=col
        )

    fig.update_xaxes(title_text="Median Income", tickprefix="$", tickformat=",", col=1)
    fig.update_xaxes(title_text="% People of Color", tickformat=".0%", col=2)
    fig.update_yaxes(title_text="Resilience (recovery rate − closure rate)", col=1)
    fig.update_layout(
        title=f"Business Resilience by Neighborhood Demographics — {naics_group}",
        width=1200, height=560,
        plot_bgcolor="white",
    )

    return fig

sector_dd = widgets.Dropdown(
    options=['All'] + sorted(combined['naics_group'].unique().tolist()),
    value='All',
    description='Sector:'
)
out = widgets.Output()

def on_change(_):
    with out:
        out.clear_output(wait=True)
        display(make_fig(sector_dd.value))

sector_dd.observe(on_change, names='value')

with out:
    display(make_fig('All'))

display(widgets.VBox([sector_dd, out]))

In [60]:
neighborhood_table.head()

,nhood,naics_group,year,closed,opened,biz_stock
0,Western Addition,Manufacturing & Industrial,2016,8.0,24.0,168
1,Western Addition,Utilities & Construction,2016,4.0,7.0,168
2,Western Addition,Retail,2016,5.0,23.0,168
3,Western Addition,Personal Services,2016,3.0,4.0,168
4,Western Addition,Food & Entertainment,2016,4.0,15.0,168


## Pre vs Post COVID Business Trajectory by Demographic Quartile

In [61]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

years = list(range(2019, 2026))

demo_cols = combined[['neighborhood', 'median_income', 'pct_poc']].drop_duplicates('neighborhood')

def make_fig(naics_group):
    if naics_group == 'All':
        nt = neighborhood_table.copy()
    else:
        nt = neighborhood_table[neighborhood_table['naics_group'] == naics_group].copy()

    # collapse across naics groups per neighborhood per year
    nt = nt.groupby(['nhood', 'year'])[['opened', 'closed']].sum().reset_index()
    nt['net_change'] = nt['opened'] - nt['closed']

    # merge demographics
    nt = nt.merge(demo_cols, left_on='nhood', right_on='neighborhood', how='left')

    # assign quartiles per neighborhood (one row per nhood for qcut)
    nhood_demo = nt[['nhood', 'median_income', 'pct_poc']].drop_duplicates('nhood')
    nhood_demo['income_quartile'] = pd.qcut(
        nhood_demo['median_income'], q=4,
        labels=['Q1 (lowest income)', 'Q2', 'Q3', 'Q4 (highest income)']
    )
    nhood_demo['poc_quartile'] = pd.qcut(
        nhood_demo['pct_poc'], q=4,
        labels=['Q1 (least POC)', 'Q2', 'Q3', 'Q4 (most POC)']
    )

    nt = nt.merge(nhood_demo[['nhood', 'income_quartile', 'poc_quartile']], on='nhood', how='left')

    income_trends = nt.groupby(['income_quartile', 'year'], observed=True)['net_change'].mean().unstack('year')
    poc_trends    = nt.groupby(['poc_quartile',    'year'], observed=True)['net_change'].mean().unstack('year')

    income_palette = ['#d6604d', '#f4a582', '#92c5de', '#2166ac']
    poc_palette    = ['#2166ac', '#92c5de', '#f4a582', '#d6604d']

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "Avg Net Business Change by Income Quartile",
            "Avg Net Business Change by % POC Quartile"
        ),
        horizontal_spacing=0.12
    )

    for i, (q_label, row) in enumerate(income_trends.iterrows()):
        fig.add_trace(go.Scatter(
            x=years, y=row.values, mode='lines+markers',
            name=str(q_label), legendgroup=f"income_{q_label}",
            line=dict(color=income_palette[i], width=2.5), marker=dict(size=7),
            hovertemplate=f"<b>{q_label}</b><br>Year: %{{x}}<br>Avg net change: %{{y:.1f}}<extra></extra>"
        ), row=1, col=1)

    for i, (q_label, row) in enumerate(poc_trends.iterrows()):
        fig.add_trace(go.Scatter(
            x=years, y=row.values, mode='lines+markers',
            name=str(q_label), legendgroup=f"poc_{q_label}",
            line=dict(color=poc_palette[i], width=2.5), marker=dict(size=7),
            hovertemplate=f"<b>{q_label}</b><br>Year: %{{x}}<br>Avg net change: %{{y:.1f}}<extra></extra>"
        ), row=1, col=2)

    for col in [1, 2]:
        fig.add_shape(type='line', x0=2020, x1=2020, y0=-50, y1=50,
                      line=dict(dash='dash', color='black', width=1), row=1, col=col)
        fig.add_annotation(x=2020, y=45, text="COVID", showarrow=False,
                           font=dict(size=9, color='black'),
                           xref=f"x{'' if col==1 else col}",
                           yref=f"y{'' if col==1 else col}")
        fig.add_shape(type='line', x0=2019, x1=2025, y0=0, y1=0,
                      line=dict(dash='dot', color='gray', width=1), row=1, col=col)

    fig.update_xaxes(tickvals=years, ticktext=[str(y) for y in years])
    fig.update_yaxes(title_text="Avg net change (openings − closings)", col=1)
    fig.update_layout(
        title=f"Pre vs Post COVID Business Trajectory — {naics_group}",
        width=1200, height=520,
        plot_bgcolor="white",
        legend=dict(orientation="v", x=1.02, y=0.5)
    )

    return fig

sector_dd = widgets.Dropdown(
    options=['All'] + sorted(neighborhood_table['naics_group'].unique().tolist()),
    value='All',
    description='Sector:'
)
out = widgets.Output()

def on_change(_):
    with out:
        out.clear_output(wait=True)
        display(make_fig(sector_dd.value))

sector_dd.observe(on_change, names='value')

with out:
    display(make_fig('All'))

display(widgets.VBox([sector_dd, out]))